# Imports

In [ ]:
# Instalar PyTorch con CUDA (primero, para evitar que sentence-transformers instale CPU)
%pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

In [ ]:
# Core libs
%pip install pandas==2.2.2 matplotlib==3.9.0 \
datasets==2.20.0 pyarrow==15.0.2 \
lime==0.2.0.1 shap==0.45.1

In [ ]:
%pip install tensorboard

# Dataset and Preprocessing

In [ ]:
import gc
import pandas as pd

from datasets import load_dataset
from datasets import Dataset

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

In [ ]:
dataset = load_dataset("imdb")
df_text_train = pd.DataFrame(dataset["train"])
df_text_test = pd.DataFrame(dataset["test"])

In [ ]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MAX_MODEL_LENGTH = 2048
MAX_LENGTH = 2048

In [ ]:
def build_prompt(review):

    prompt = f"""Review: {review}

Sentiment:"""

    return prompt


def build_full_text(review, label):

    sentiment = (
        "positive"
        if label == 1
        else "negative"
    )

    full_text = f"""Review: {review}

Sentiment: {sentiment}"""

    return full_text


def preprocess_dataset(dataset, tokenizer):

    input_ids_list = []
    attention_masks_list = []
    labels_list = []
    true_labels_list = []

    for example in dataset:

        review = example["text"]
        label = example["label"]

        prompt = build_prompt(review)
        full_text = build_full_text(review, label)

        # =========================
        # TOKENIZACIÓN
        # =========================

        full = tokenizer(
            full_text,
            add_special_tokens=False
        )

        input_ids = full["input_ids"]

        # attention mask inicial
        attention_mask = [1] * len(input_ids)

        # =========================
        # LABELS (SIN prompt_len)
        # =========================

        labels = []

        # reconstruimos prompt tokenizado directamente dentro del full_text
        prompt_text = build_prompt(review)

        prompt_ids = tokenizer(
            prompt_text,
            add_special_tokens=False
        )["input_ids"]

        # ⚠️ no asumimos alineación por slicing, buscamos corte seguro por longitud
        prompt_len = len(prompt_ids)

        for i, tok in enumerate(input_ids):

            if i < prompt_len:
                labels.append(-100)
            else:
                labels.append(tok)

        # =========================
        # PADDING / TRUNCATION UNIFORME
        # =========================

        if len(input_ids) > MAX_LENGTH:

            input_ids = input_ids[:MAX_LENGTH]
            attention_mask = attention_mask[:MAX_LENGTH]
            labels = labels[:MAX_LENGTH]

        else:

            pad_len = MAX_LENGTH - len(input_ids)

            input_ids += [tokenizer.pad_token_id] * pad_len
            attention_mask += [0] * pad_len
            labels += [-100] * pad_len

        # =========================
        # TRUE LABEL EXPLÍCITO
        # =========================

        true_label = "positive" if label == 1 else "negative"

        # =========================
        # STORE
        # =========================

        input_ids_list.append(input_ids)
        attention_masks_list.append(attention_mask)
        labels_list.append(labels)
        true_labels_list.append(true_label)

    return Dataset.from_dict({
        "input_ids": input_ids_list,
        "attention_mask": attention_masks_list,
        "labels": labels_list,
        "true_label": true_labels_list
    })

In [ ]:
# Model tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
def fits_model(example):

    text = build_full_text(
        example["text"],
        example["label"]
    )

    tokens = tokenizer(
        text,
        add_special_tokens=False
    )["input_ids"]

    return len(tokens) <= MAX_MODEL_LENGTH

In [ ]:
split = dataset["train"].train_test_split(
    test_size=0.1,
    seed=42
)

train_raw = split["train"]
val_raw = split["test"]
test_raw = dataset["test"]

In [ ]:
# Reviews exceeding model context are removed from the dataset.

filtered_train = train_raw.filter(
    fits_model
)

filtered_val = val_raw.filter(
    fits_model
)

filtered_test = test_raw.filter(
    fits_model
)

In [ ]:
train_dataset = preprocess_dataset(filtered_train, tokenizer)
val_dataset = preprocess_dataset(filtered_val, tokenizer)
test_dataset = preprocess_dataset(filtered_test, tokenizer)

In [ ]:
train_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels",
        "true_label"
    ]
)

val_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels",
        "true_label"
    ]
)

test_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels",
        "true_label"
    ]
)

# Student definition

In [ ]:
# LLM Model download
teacher_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME
)

In [ ]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

In [ ]:
teacher_model.to(device)

In [ ]:
for name, param in teacher_model.named_parameters():
    print(name, param.shape)

In [ ]:
from transformers import AutoConfig

# cargar config del teacher
config = AutoConfig.from_pretrained(MODEL_NAME)

# reducir capas si querés student 11 capas
config.num_hidden_layers = 11

# crear student RANDOM
student_model = AutoModelForCausalLM.from_config(config)

In [ ]:
student_model.to(device)

# Student training

In [ ]:
BATCH_SIZE = 1

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [ ]:
def save_checkpoint(model, optimizer, epoch, global_step, path="checkpoint.pt"):

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch": epoch,
            "global_step": global_step
        },
        path
    )

def load_checkpoint(model, optimizer, path="checkpoint.pt"):

    ckpt = torch.load(path, map_location=device)

    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])

    return ckpt["epoch"], ckpt["global_step"]

In [ ]:
# =========================
# CONFIG DISTILLATION
# =========================
temperature = 2.0
alpha = 0.5
beta = 0.5
lr = 1e-4

EPOCHS = 3

teacher_model.eval()
student_model.train()

for p in teacher_model.parameters():
    p.requires_grad = False

optimizer = torch.optim.AdamW(
    student_model.parameters(),
    lr=lr
)

In [ ]:
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter(
    log_dir="./runs/student_local"
)

In [ ]:
best_val_loss = float("inf")
eval_every = 1000
save_every = 2000
start_epoch = 0
global_step = 0
max_val_steps = 100

In [ ]:
positive_id = tokenizer.decode("positive")
negative_id = tokenizer.decode("negative")

In [ ]:
try:
    start_epoch, global_step = load_checkpoint(
        student_model,
        optimizer,
        "checkpoints/student_local_checkpoint.pt"
    )

    print(f"[RESUME] Epoch {start_epoch} | Global step {global_step}")

except:
    print("[START] Training from scratch")

# =========================
# INIT LOSS
# =========================

student_model.eval()

with torch.no_grad():
    batch = next(iter(train_loader))

    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    teacher_outputs = teacher_model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

    student_outputs = student_model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels
    )

    ce_loss = student_outputs.loss

    kd_global = F.kl_div(
        F.log_softmax(student_outputs.logits / temperature, dim=-1),
        F.softmax(teacher_outputs.logits / temperature, dim=-1),
        reduction="batchmean"
    ) * (temperature ** 2)

    # =========================
    # KD OBJ (SCALED)
    # =========================

    student_obj = student_outputs.logits[:, :, [positive_id, negative_id]]
    teacher_obj = teacher_outputs.logits[:, :, [positive_id, negative_id]]

    kd_obj_raw = F.kl_div(
        F.log_softmax(student_obj / temperature, dim=-1),
        F.softmax(teacher_obj / temperature, dim=-1),
        reduction="batchmean"
    ) * (temperature ** 2)

    # 🔥 NORMALIZACIÓN CRÍTICA
    kd_obj = kd_obj_raw / 2.0  # tamaño del subespacio

    init_loss = alpha * ce_loss + (1 - alpha) * kd_global + beta * kd_obj

    print(
        f"[INIT LOSS] Total={init_loss.item():.4f} | "
        f"CE={ce_loss.item():.4f} | "
        f"KDg={kd_global.item():.4f} | "
        f"KDobj={kd_obj.item():.4f}"
    )

student_model.train()

# =========================
# TENSORBOARD SETUP
# =========================

writer.add_text("config/alpha", str(alpha))
writer.add_text("config/beta", str(beta))
writer.add_text("config/temperature", str(temperature))

# =========================
# TRAINING
# =========================

for epoch in range(start_epoch, EPOCHS):

    student_model.train()

    total_loss = 0
    total_ce = 0
    total_kd_global = 0
    total_kd_obj = 0

    for step, batch in enumerate(train_loader):

        global_step += 1

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # =========================
        # TEACHER
        # =========================

        with torch.no_grad():
            teacher_outputs = teacher_model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        # =========================
        # STUDENT
        # =========================

        student_outputs = student_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        ce_loss = student_outputs.loss

        kd_global = F.kl_div(
            F.log_softmax(student_outputs.logits / temperature, dim=-1),
            F.softmax(teacher_outputs.logits / temperature, dim=-1),
            reduction="batchmean"
        ) * (temperature ** 2)

        student_obj = student_outputs.logits[:, :, [positive_id, negative_id]]
        teacher_obj = teacher_outputs.logits[:, :, [positive_id, negative_id]]

        kd_obj_raw = F.kl_div(
            F.log_softmax(student_obj / temperature, dim=-1),
            F.softmax(teacher_obj / temperature, dim=-1),
            reduction="batchmean"
        ) * (temperature ** 2)

        kd_obj = kd_obj_raw / 2.0

        # =========================
        # LOSS FINAL
        # =========================

        loss = (
            alpha * ce_loss +
            (1 - alpha) * kd_global +
            beta * kd_obj
        )

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        # =========================
        # ACCUMULATORS
        # =========================

        total_loss += loss.item()
        total_ce += ce_loss.item()
        total_kd_global += kd_global.item()
        total_kd_obj += kd_obj.item()

        # =========================
        # TENSORBOARD LOGGING
        # =========================

        writer.add_scalar("Loss/train_total", loss.item(), global_step)
        writer.add_scalar("Loss/train_ce", ce_loss.item(), global_step)
        writer.add_scalar("Loss/train_kd_global", kd_global.item(), global_step)
        writer.add_scalar("Loss/train_kd_obj", kd_obj.item(), global_step)

        # ratios (muy útiles para análisis en tesis)
        writer.add_scalar(
            "Loss/ratio_kd_obj_over_global",
            (kd_obj.item() / (kd_global.item() + 1e-8)),
            global_step
        )

        if global_step % 50 == 0:
            print(
                f"Epoch {epoch} | Step {step} | Global {global_step} | "
                f"Loss {total_loss/(step+1):.4f} | "
                f"CE {total_ce/(step+1):.4f} | "
                f"KDg {total_kd_global/(step+1):.4f} | "
                f"KDobj {total_kd_obj/(step+1):.4f}"
            )

        # =========================
        # VALIDATION
        # =========================

        if global_step % eval_every == 0:

            student_model.eval()

            val_loss = 0
            val_steps = 0

            with torch.no_grad():

                for val_batch in val_loader:

                    val_steps += 1

                    input_ids = val_batch["input_ids"].to(device)
                    attention_mask = val_batch["attention_mask"].to(device)
                    labels = val_batch["labels"].to(device)

                    teacher_outputs = teacher_model(
                        input_ids=input_ids,
                        attention_mask=attention_mask
                    )

                    student_outputs = student_model(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels
                    )

                    ce_loss = student_outputs.loss

                    kd_global = F.kl_div(
                        F.log_softmax(student_outputs.logits / temperature, dim=-1),
                        F.softmax(teacher_outputs.logits / temperature, dim=-1),
                        reduction="batchmean"
                    ) * (temperature ** 2)

                    student_obj = student_outputs.logits[:, :, [positive_id, negative_id]]
                    teacher_obj = teacher_outputs.logits[:, :, [positive_id, negative_id]]

                    kd_obj = F.kl_div(
                        F.log_softmax(student_obj / temperature, dim=-1),
                        F.softmax(teacher_obj / temperature, dim=-1),
                        reduction="batchmean"
                    ) * (temperature ** 2)

                    kd_obj = kd_obj / 2.0

                    batch_loss = (
                        alpha * ce_loss +
                        (1 - alpha) * kd_global +
                        beta * kd_obj
                    )

                    val_loss += batch_loss.item()

                    if val_steps >= max_val_steps:
                        break

            val_loss /= val_steps

            writer.add_scalar("Loss/validation", val_loss, global_step)

            print(f"\n[VAL LOSS] {val_loss:.4f}\n")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0

                torch.save(
                    student_model.state_dict(),
                    "./checkpoints/best_student_model.pt"
                )

                print("[BEST MODEL SAVED]")

            else:
                patience_counter += 1

            save_checkpoint(
                student_model,
                optimizer,
                epoch,
                global_step,
                "checkpoints/student_checkpoint.pt"
            )

        # =========================
        # CLEANUP
        # =========================

        del (
            input_ids,
            attention_mask,
            labels,
            teacher_outputs,
            student_outputs,
            ce_loss,
            kd_global,
            kd_obj,
            kd_obj_raw,
            loss
        )

    gc.collect()

writer.flush()

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# =========================
# TARGET TOKENS
# =========================

positive_id = tokenizer.encode(
    " positive",
    add_special_tokens=False
)[0]

negative_id = tokenizer.encode(
    " negative",
    add_special_tokens=False
)[0]

# =========================
# TEST EVALUATION
# =========================

teacher_model.eval()
student_model.eval()

y_true = []
y_pred_student = []

kl_total = 0.0
n_batches = 0

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        true_labels = batch["true_label"]

        # =========================
        # TEACHER
        # =========================
        teacher_outputs = teacher_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # =========================
        # STUDENT
        # =========================
        student_outputs = student_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # =========================
        # POSICIÓN CORRECTA
        # =========================
        last_idx = attention_mask.sum(dim=1) - 1

        teacher_logits = teacher_outputs.logits[
            torch.arange(input_ids.size(0), device=device),
            last_idx
        ]

        student_logits = student_outputs.logits[
            torch.arange(input_ids.size(0), device=device),
            last_idx
        ]

        # =========================
        # CLASSIFICATION (student)
        # =========================
        student_positive = student_logits[:, positive_id]
        student_negative = student_logits[:, negative_id]

        preds = student_positive > student_negative

        for p, t in zip(preds, true_labels):

            y_pred_student.append(
                "positive" if p.item() else "negative"
            )
            y_true.append(t)

        # =========================
        # KL DIVERGENCE (student vs teacher)
        # =========================

        T = 2.0  # podés usar el mismo temperature del training

        kl = F.kl_div(
            F.log_softmax(student_logits / T, dim=-1),
            F.softmax(teacher_logits / T, dim=-1),
            reduction="batchmean"
        ) * (T ** 2)

        kl_total += kl.item()
        n_batches += 1

# =========================
# FINAL METRICS
# =========================

kl_avg = kl_total / n_batches

print("\n===== TEST RESULTS =====")
print("KL(student || teacher):", kl_avg)

# =========================
# METRICS
# =========================

test_accuracy = accuracy_score(
    y_true,
    y_pred_student
)

test_precision = precision_score(
    y_true,
    y_pred_student,
    average="macro",
    zero_division=0
)

test_recall = recall_score(
    y_true,
    y_pred_student,
    average="macro",
    zero_division=0
)

test_f1 = f1_score(
    y_true,
    y_pred_student,
    average="macro",
    zero_division=0
)

cm = confusion_matrix(
    y_true,
    y_pred_student,
    labels=[
        "positive",
        "negative"
    ]
)

In [ ]:
# =========================
# PRINT
# =========================

print("\n=========================")
print("[TEST RESULTS]")
print(f"Accuracy : {test_accuracy:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall   : {test_recall:.4f}")
print(f"F1 Score : {test_f1:.4f}")
print("=========================\n")

print("[CONFUSION MATRIX]")
print(cm)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 5))

im = ax.imshow(cm, cmap="Reds", vmax=16000)

# labels
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["positive", "negative"])
ax.set_yticklabels(["positive", "negative"])

ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title("Confusion Matrix")

# valores dentro de las celdas
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j],
                ha="center", va="center",
                color="black")

plt.colorbar(im)
plt.show()

In [ ]:
# =========================
# TENSORBOARD
# =========================

writer.add_scalar(
    "Test/accuracy",
    test_accuracy,
    global_step
)

writer.add_scalar(
    "Test/precision",
    test_precision,
    global_step
)

writer.add_scalar(
    "Test/recall",
    test_recall,
    global_step
)

writer.add_scalar(
    "Test/f1",
    test_f1,
    global_step
)

writer.add_scalar(
    "Test/kl_divergence",
    kl_avg,
    global_step
)

writer.close()